### Base Image

In [9]:
import os
os.makedirs("docker-file-pytorch", exist_ok=True)

In [33]:
%%bash
cat <<'EOF' > docker-file-pytorch/Dockerfile
FROM pytorch/pytorch:2.14.0-cuda13.2-cudnn9-runtime

RUN printf "[global]\nbreak-system-packages = true\n" > /etc/pip.conf

RUN apt-get update \
    && apt-get install -y --no-install-recommends git \
    && rm -rf /var/lib/apt/lists/*

RUN pip install --no-cache-dir \
    "numpy==1.26.4" \
    mxnet \
    torchvision \
    timm \
    tensorboard \
    easydict \
    onnx \
    scikit-learn \
    opencv-python \
    google-cloud-storage
EOF

In [41]:
# !gcloud artifacts repositories list --project=$(gcloud config get-value project)

In [38]:
# !gcloud artifacts repositories create rf-pytorch \
#     --repository-format=docker \
#     --location=us-central1 \
#     --description="PyTorch Docker images"

In [39]:
# !gcloud builds submit docker-file-pytorch \
#     --tag us-central1-docker.pkg.dev/$(gcloud config get-value project)/rf-pytorch/rf-arcface:latest \
#     --region=us-central1

### Kubeflow

In [21]:
from kfp.dsl import component
from kfp import compiler, dsl
from kfp.dsl import Output, Dataset, Input
import os
from dotenv import load_dotenv
from google.cloud import aiplatform
load_dotenv(override=True)

True

In [22]:
PROJECT_ID = os.environ["PROJECT_ID"]
PIPELINE_ROOT = os.environ["PIPELINE_ROOT"]

In [42]:
IMAGE_URI = f"us-central1-docker.pkg.dev/{PROJECT_ID}/rf-pytorch/rf-arcface:latest"

In [43]:
@dsl.component(
    base_image=IMAGE_URI
)
def train_model():
    import sys
    import numpy as np
    import torch

    np.long = np.int_
    np.ulong = np.uint

    if not hasattr(np, "bool"):
        np.bool = bool

    import mxnet as mx

    print("Python:", sys.version)
    print("NumPy:", np.__version__)
    print("PyTorch:", torch.__version__)
    print("MXNet:", mx.__version__)
    print("CUDA:", torch.version.cuda)
    print("CUDA available:", torch.cuda.is_available())
    print("GPU count:", torch.cuda.device_count())

In [25]:
@dsl.pipeline(name="Pipeline")
def pipeline():
    train_task = train_model()
    
    train_task.set_cpu_limit("4")
    train_task.set_memory_limit("16Gi")
    
    train_task.set_accelerator_type("NVIDIA_TESLA_T4")
    train_task.set_accelerator_limit(1)
    
compiler.Compiler().compile(
    pipeline_func=pipeline,
    package_path="pipeline.yaml",
)

In [26]:
job = aiplatform.PipelineJob(
    display_name="pipeline",
    template_path="pipeline.yaml",
    pipeline_root=PIPELINE_ROOT,
    parameter_values={},
    location="us-central1",
)

In [27]:
SERVICE_ACCOUNT = os.environ["SERVICE_ACCOUNT"]
job.submit(service_account = SERVICE_ACCOUNT)

Creating PipelineJob
PipelineJob created. Resource name: projects/344969539300/locations/us-central1/pipelineJobs/pipeline-20261020111118
To use this PipelineJob in another session:
pipeline_job = aiplatform.PipelineJob.get('projects/344969539300/locations/us-central1/pipelineJobs/pipeline-20261020111118')
View Pipeline Job:
https://console.cloud.google.com/vertex-ai/locations/us-central1/pipelines/runs/pipeline-20261020111118?project=344969539300
